In [3]:
!pip install mlflow
!pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   -- ------------------------------------- 7.3/101.7 MB 50.2 MB/s eta 0:00:02
   -------- ------------------------------- 21.5/101.7 MB 61.7 MB/s eta 0:00:02
   -------------- ------------------------- 36.2/101.7 MB 63.9 MB/s eta 0:00:02
   -------------------- ------------------- 50.9/101.7 MB 66.0 MB/s eta 0:00:01
   ------------------------- -------------- 65.8/101.7 MB 66.6 MB/s eta 0:00:01
   ------------------------------- -------- 80.5/101.7 MB 66.7 MB/s eta 0:00:01
   ------------------------------------- -- 95.2/101.7 MB 67.5 MB/s eta 0:00:01
   --------------------------------------  101.4/101.7 MB 67.5 MB/s eta 0:00:01
   --------------------------------------- 101.7/101.7 MB 58.5 MB/s eta 0:00:00


In [2]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [3]:
X,y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8,
                          weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100], dtype=int64))

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

### Experiment 1 - Logistic regression

In [5]:
#Define the model hyperparameters
params = {
    "solver":"lbfgs",
    "max_iter":1000,
    "multi_class":"auto",
    "random_state": 8888
}

#train the model
lr = LogisticRegression(**params)
lr.fit(X_train, y_train)

#predict on the test set
y_pred = lr.predict(X_test)

report = classification_report(y_test, y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.95      0.97      0.96       270
           1       0.62      0.50      0.56        30

    accuracy                           0.92       300
   macro avg       0.79      0.73      0.76       300
weighted avg       0.91      0.92      0.92       300



In [6]:
report_dict = classification_report(y_test, y_pred, output_dict= True)
report_dict

{'0': {'precision': 0.9456521739130435,
  'recall': 0.9666666666666667,
  'f1-score': 0.9560439560439561,
  'support': 270.0},
 '1': {'precision': 0.625,
  'recall': 0.5,
  'f1-score': 0.5555555555555556,
  'support': 30.0},
 'accuracy': 0.92,
 'macro avg': {'precision': 0.7853260869565217,
  'recall': 0.7333333333333334,
  'f1-score': 0.7557997557997558,
  'support': 300.0},
 'weighted avg': {'precision': 0.9135869565217392,
  'recall': 0.92,
  'f1-score': 0.9159951159951161,
  'support': 300.0}}

In [7]:
report_dict['accuracy']

0.92

In [8]:
import mlflow

In [ ]:

mlflow.set_tracking_uri(uri="http://127.0.0.1:5000/") #as colab is running in the cloud is difficult to connect it to a local host
mlflow.set_experiment("First experiment")

with mlflow.start_run():
  mlflow.log_params(params)
  mlflow.log_metrics({
      'accuracy': 0.92,
      'recall_class_0': report_dict['0']['recall'],
      'recall_class_1': report_dict['1']['recall'],
      'f1_score_macro': report_dict['macro avg']['f1-score']
  })
  mlflow.sklearn.log_model(lr, "Logistic Regression")

### Experiment 2 - random forest

In [10]:
rf_clf = RandomForestClassifier(n_estimators = 30, max_depth = 3)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.97      0.99      0.98       270
           1       0.91      0.70      0.79        30

    accuracy                           0.96       300
   macro avg       0.94      0.85      0.89       300
weighted avg       0.96      0.96      0.96       300



### Experiment 3 - xgboost

In [11]:
xgb_clf = XGBClassifier(use_label_encoder = False, eval_metric='logloss')
xgb_clf.fit(X_train, y_train)
y_pred_xgb = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       270
           1       0.96      0.80      0.87        30

    accuracy                           0.98       300
   macro avg       0.97      0.90      0.93       300
weighted avg       0.98      0.98      0.98       300



### Experiment 4 - handle class imbalance using SMOTEtomek and then train xgboost

In [13]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)

np.unique(y_train_res, return_counts = True)

(array([0, 1]), array([619, 619], dtype=int64))

In [14]:
xgb_clf_res = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf_res.fit(X_train_res, y_train_res)
y_pred_xgb_res = xgb_clf_res.predict(X_test)
print(classification_report(y_test, y_pred_xgb_res))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       270
           1       0.81      0.83      0.82        30

    accuracy                           0.96       300
   macro avg       0.89      0.91      0.90       300
weighted avg       0.96      0.96      0.96       300



# track experiments Using Mlflow

In [21]:
models = [
    (
        "Logistic Regression",
        {"C":1, "solver":'liblinear'},
        LogisticRegression(),
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest",
        {"n_estimators":30, "max_depth": 3},
        RandomForestClassifier(),
        (X_train, y_train),
        (X_test, y_test)   
    ),
    (
        "XGBClassifier",
        {"use_label_encoder":False, "eval_metric":'logloss'},
        XGBClassifier(),
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier with SMOTE",
        {"use_label_encoder":False, "eval_metric":'logloss'},
        XGBClassifier(),
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [22]:
reports =[]

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train= train_set[1]
    X_test = test_set[0]
    y = test_set[1]
    
    model.set_params(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

In [34]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Anomaly Detection v3")

for i, element in enumerate(models):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = reports[i]
  
    with mlflow.start_run(run_name=model_name):
        mlflow.log_params(params)
        mlflow.log_metrics({
                'accuracy': report['accuracy'],
                'recall_class_1': report['1']['recall'],
                'recall_class_0': report['0']['recall'],
                'f1_score_macro': report['macro avg']['f1-score']
            })
    
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model,"model")
        else:
            mlflow.sklearn.log_model(model,"model")

2026/04/07 17:16:12 INFO mlflow.tracking.fluent: Experiment with name 'Anomaly Detection v3' does not exist. Creating a new experiment.
2026/04/07 17:16:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/07 17:16:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/07 17:16:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/07 17:16:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during des

🏃 View run Logistic Regression at: http://localhost:5000/#/experiments/4/runs/5f257529f03140f9a34b102f6b131fbd
🧪 View experiment at: http://localhost:5000/#/experiments/4


2026/04/07 17:16:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: http://localhost:5000/#/experiments/4/runs/64308d627c524cca812940c089f8a132
🧪 View experiment at: http://localhost:5000/#/experiments/4


2026/04/07 17:16:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://localhost:5000/#/experiments/4/runs/48de6296f23c4a0db61232dd8ec79f50
🧪 View experiment at: http://localhost:5000/#/experiments/4
🏃 View run XGBClassifier with SMOTE at: http://localhost:5000/#/experiments/4/runs/5dac479d7f6342bab2c8358e0fe51c59
🧪 View experiment at: http://localhost:5000/#/experiments/4


# register the model


In [37]:
artifact_name = "model"       
registry_name = "XGB-Smote"   # nombre que tendrá en el registry

run_id = input("Enter Run ID:")
model_uri = f"runs:/{run_id}/{artifact_name}"

result = mlflow.register_model(model_uri, registry_name)

Enter Run ID: 5dac479d7f6342bab2c8358e0fe51c59


Registered model 'XGB-Smote' already exists. Creating a new version of this model...
2026/04/07 17:22:16 WARNING mlflow.tracking._model_registry.fluent: Run with id 5dac479d7f6342bab2c8358e0fe51c59 has no artifacts at artifact path 'model', registering model based on models:/m-623577b56978420191cd2636a675fe0d instead
2026/04/07 17:22:16 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB-Smote, version 1
Created version '1' of model 'XGB-Smote'.


# Load the model

In [38]:
model_version = 1
model_uri = f"models:/{model_name}/{model_version}"

loaded_model = mlflow.xgboost.load_model(model_uri)
loaded_model.predict(X_test) 
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])

In [39]:
model_version = 1
model_uri = f"models:/{model_name}@challenger"

loaded_model = mlflow.xgboost.load_model(model_uri)
loaded_model.predict(X_test) 
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])

# Transitioning from dev to prod

In [44]:
dev_model_uri = f"models:/{model_name}@challenger"
prod_model = 'anomaly-detection-prod'

client = mlflow.MlflowClient()
client.copy_model_version(src_model_uri = dev_model_uri, dst_name=prod_model)

Successfully registered model 'anomaly-detection-prod'.
Copied version '1' of model 'XGB-Smote' to version '1' of model 'anomaly-detection-prod'.


<ModelVersion: aliases=[], creation_timestamp=1775601292885, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='This model was trained on oversampled dataset and the classifier is XGB', last_updated_timestamp=1775601292885, metrics=None, model_id=None, name='anomaly-detection-prod', params=None, run_id='5dac479d7f6342bab2c8358e0fe51c59', run_link='', source='models:/XGB-Smote/1', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>